# 5.7 阶段五：mask

到目前为止我们的 FA 只会「全关注」。真实模型需要掩码：因果掩码（只看过去）、带状掩码（只看邻域）、padding 掩码（跳过补齐区）。阶段五给四拍循环加上 mask 支持。

## 1. mask 在哪一拍生效

mask 作用在**分数矩阵 S 上**（V1 拍内）：把不允许关注的位置的分数置为 $-\infty$，softmax 后这些位置的权重就是 0。

```text
C1: S = Q·Kᵀ·scale
V1: S' = S + mask        ← mask 在这里生效（加 -inf）
    P = FlashSoftmax(S')
C2: O = P·V
V2: update / 归一化
```

工程上 mask 通常**预计算成与 S 同形状的布尔/数值矩阵**，V1 里一条加法指令叠加即可；也可以用 sparsemode 让 Kernel 在编解码时直接跳过整块（见下）。

## 2. sparsemode：块级跳过

因果掩码有个漂亮的性质（第 2 章 2.5 节）：分块后**整块整块地跳过未来块**。昇腾 FA 算子把常见掩码形态编成了 **sparsemode 枚举**，本阶段启用 **sparsemode3**（因果掩码的标准模式）：

| sparsemode | 语义（示意） |
|--|--|
| 0 | 不做稀疏，全量计算 |
| 1/2 | 带状/自定义稀疏描述 |
| **3** | **因果掩码：preTokens/nextTokens 描述的边界 + 块级跳过** |

sparsemode3 下 Tiling 和 Kernel 的联动：

1. **Tiling**：按 `preTokens`/`nextTokens` 计算每个 (S1 块, S2 块) 是否需要计算，生成每个 S1 块的 S2 起止范围（块级跳过表）；
2. **Kernel**：内层 S2 循环直接从起点跑到终点，**跳过的块既不算也不搬**——计算量近似减半（第 2 章的结论在工程上落地）。

## 3. 尾块边界：行级掩码

块级跳过只解决「整块跳过」；S2 尾块内部（或 preTokens 边界落在块中间时）需要**行级/元素级掩码**：

```cpp
// V1 拍内：先按块级判断，再按行级叠加
// s2Start..s2End 是本块的有效区间（sparsemode3 计算出）
for (int row = 0; row < s1Block; row++) {
    // 因果边界：第 row 行只允许 [0, row + preTokens] 范围内的列
    int validEnd = min(s2BlockEnd, rowOffset + row + preTokens);
    // 有效区间外的分数置 -inf（select / 比较 + 加法实现）
}
```

> 实现提示：逐元素掩码用 Vector 的比较+选择指令（如 `Compare` + `Select`）在 UB 上完成，避免 Scalar 逐个处理。

## 4. 对 Tiling 的影响

mask 引入后 Tiling 多了两个产出：每 S1 块的**S2 循环起止表**和 sparsemode 本身（进 TilingKey，编译期隔离 mask 代码段——第 3 章 3.4 节原则二）。

## 小测验

1. mask 在四拍循环的哪一拍、以什么方式生效？
2. sparsemode3 的「块级跳过」省掉的是什么？为什么因果掩码能整块跳？
3. 边界落在块中间时为什么还需要行级掩码？用 Vector 指令而不是 Scalar 逐个处理的原因？